In [ ]:
import os
import re
import pandas as pd
import numpy as np
RAW_PATH = "data/raw/movies.csv"
RAW_PATH = r"C:\Users\Saeed\Desktop\neural_network_project\data\raw\movies.csv"
PROCESSED_DIR = r"C:\Users\Saeed\Desktop\neural_network_project\data\processed"
MOVIELENS_DIR = r"C:\Users\Saeed\Desktop\neural_network_project\data\raw\movielens"

os.makedirs(PROCESSED_DIR, exist_ok=True)
print("Libraries loaded OK")

Libraries loaded OK


In [ ]:
df = pd.read_csv(RAW_PATH)
print(f"Loaded: {len(df)} movies")
print(df.shape)
df.head(3)

Loaded: 9988 movies
(9988, 26)


,movie_id,title,original_title,release_date,release_year,runtime,status,language,overview,tagline,...,collection,poster_path,backdrop_path,vote_average,vote_count,popularity,budget,revenue,tmdb_url,imdb_id
0,983044,The Arctic Convoy,Konvoi,2023-12-25,2023,109,Released,no,"In 1942, a convoy of 35 civilian ships, carryi...",Courage becomes their only lifeline.,...,NaN,https://image.tmdb.org/t/p/w500/7qtd6xschdz7GB...,https://image.tmdb.org/t/p/w780/4BXIBNeYpKkOrY...,6.948,145,1.8276,6600000,3637940,https://www.themoviedb.org/movie/983044,tt27724113
1,5,Four Rooms,Four Rooms,1995-12-09,1995,98,Released,en,It's Ted the Bellhop's first night on the job....,Twelve outrageous guests. Four scandalous requ...,...,NaN,https://image.tmdb.org/t/p/w500/75aHn1NOYXh4M7...,https://image.tmdb.org/t/p/w780/c1BaOxC8bo5ACF...,5.899,2827,3.5756,4000000,4257354,https://www.themoviedb.org/movie/5,tt0113101
2,11,Star Wars,Star Wars,1977-05-25,1977,121,Released,en,Princess Leia is captured and held hostage by ...,"A long time ago in a galaxy far, far away...",...,Star Wars Collection,https://image.tmdb.org/t/p/w500/6FfCtAuVAW8XJj...,https://image.tmdb.org/t/p/w780/yUiXA68FfQeA8c...,8.204,22233,25.7556,11000000,775398007,https://www.themoviedb.org/movie/11,tt0076759


## 📊 Load Raw Dataset

Load the TMDB movies dataset and display basic information.

In [ ]:
# إزالة duplicates
df = df.drop_duplicates(subset="movie_id").reset_index(drop=True)

# تنظيف النصوص
df["overview"]  = df["overview"].fillna("").str.strip()
df["title"]     = df["title"].fillna("Unknown")
df["genres"]    = df["genres"].fillna("")
df["cast"]      = df["cast"].fillna("")
df["director"]  = df["director"].fillna("Unknown")
df["keywords"]  = df["keywords"].fillna("")
df["tagline"]   = df["tagline"].fillna("")

# تنظيف الأرقام
df["vote_average"] = pd.to_numeric(df["vote_average"], errors="coerce").fillna(0)
df["vote_count"]   = pd.to_numeric(df["vote_count"],   errors="coerce").fillna(0)
df["popularity"]   = pd.to_numeric(df["popularity"],   errors="coerce").fillna(0)

# فلتر جودة
df = df[
    (df["overview"].str.len() > 20) &
    (df["vote_count"] >= 50)
].reset_index(drop=True)

print(f"After cleaning: {len(df)} movies")
df.info()

After cleaning: 9988 movies
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9988 entries, 0 to 9987
Data columns (total 26 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   movie_id              9988 non-null   int64  
 1   title                 9988 non-null   object 
 2   original_title        9988 non-null   object 
 3   release_date          9988 non-null   object 
 4   release_year          9988 non-null   int64  
 5   runtime               9988 non-null   int64  
 6   status                9988 non-null   object 
 7   language              9988 non-null   object 
 8   overview              9988 non-null   object 
 9   tagline               9988 non-null   object 
 10  genres                9988 non-null   object 
 11  cast                  9988 non-null   object 
 12  director              9988 non-null   object 
 13  writer                9848 non-null   object 
 14  keywords              9988 non-null   object

In [ ]:
print("=== Dataset Overview ===")
print(f"Total movies     : {len(df)}")
print(f"With poster URL  : {df['poster_path'].notna().sum()}")
print(f"With overview    : {(df['overview'].str.len() > 20).sum()}")
print(f"Avg vote_average : {df['vote_average'].mean():.2f}")
print(f"Avg vote_count   : {df['vote_count'].mean():.0f}")
print(f"\nTop 5 genres:")
genres_all = df["genres"].str.split("|").explode()
print(genres_all.value_counts().head())

=== Dataset Overview ===
Total movies     : 9988
With poster URL  : 9988
With overview    : 9988
Avg vote_average : 6.66
Avg vote_count   : 2013

Top 5 genres:
genres
Drama        4378
Comedy       3086
Action       3067
Thriller     2754
Adventure    1904
Name: count, dtype: int64


In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["content_soup"] = (
    df["overview"].apply(clean_text) + " " +
    df["tagline"].apply(clean_text) + " " +
    df["genres"].str.replace("|", " ") + " " +
    df["keywords"].str.replace("|", " ")
)

content_df = df[["movie_id", "title", "overview", "content_soup", "poster_path"]].copy()
content_df.to_csv(f"{PROCESSED_DIR}/content_based.csv", index=False)
print(f"content_based.csv saved — {len(content_df)} rows")
content_df.head(3)

content_based.csv saved — 9988 rows


,movie_id,title,overview,content_soup,poster_path
0,983044,The Arctic Convoy,"In 1942, a convoy of 35 civilian ships, carryi...",in 1942 a convoy of 35 civilian ships carrying...,https://image.tmdb.org/t/p/w500/7qtd6xschdz7GB...
1,5,Four Rooms,It's Ted the Bellhop's first night on the job....,it s ted the bellhop s first night on the job ...,https://image.tmdb.org/t/p/w500/75aHn1NOYXh4M7...
2,11,Star Wars,Princess Leia is captured and held hostage by ...,princess leia is captured and held hostage by ...,https://image.tmdb.org/t/p/w500/6FfCtAuVAW8XJj...


In [ ]:
def split_col(val):
    if pd.isna(val) or val == "":
        return []
    return [x.strip().lower().replace(" ", "_") for x in str(val).split("|")]

df["genres_list"]    = df["genres"].apply(split_col)
df["cast_list"]      = df["cast"].apply(split_col)
df["keywords_list"]  = df["keywords"].apply(split_col)
df["director_clean"] = df["director"].str.lower().str.replace(" ", "_").fillna("")

def make_soup(row):
    parts = (
        row["genres_list"] +
        row["cast_list"][:3] +
        [row["director_clean"]] * 3 +
        row["keywords_list"][:5]
    )
    return " ".join(parts)

df["metadata_soup"] = df.apply(make_soup, axis=1)

metadata_df = df[["movie_id", "title", "genres_list", "cast_list",
                   "director_clean", "keywords_list", "metadata_soup", "poster_path"]].copy()
metadata_df.to_csv(f"{PROCESSED_DIR}/metadata.csv", index=False)
print(f"metadata.csv saved — {len(metadata_df)} rows")
metadata_df.head(3)

metadata.csv saved — 9988 rows


,movie_id,title,genres_list,cast_list,director_clean,keywords_list,metadata_soup,poster_path
0,983044,The Arctic Convoy,"[war, drama]","[tobias_santelmann, anders_baasmo_christiansen...",henrik_martin_dahlsbakken,"[world_war_ii, based_on_true_story, struggle_f...",war drama tobias_santelmann anders_baasmo_chri...,https://image.tmdb.org/t/p/w500/7qtd6xschdz7GB...
1,5,Four Rooms,[comedy],"[tim_roth, jennifer_beals, david_proval, ione_...",robert_rodriguez,"[hotel, new_year's_eve, witch, bet, sperm, hot...",comedy tim_roth jennifer_beals david_proval ro...,https://image.tmdb.org/t/p/w500/75aHn1NOYXh4M7...
2,11,Star Wars,"[adventure, action, science_fiction]","[mark_hamill, harrison_ford, carrie_fisher, pe...",george_lucas,"[empire, galaxy, rebellion, android, hermit, s...",adventure action science_fiction mark_hamill h...,https://image.tmdb.org/t/p/w500/6FfCtAuVAW8XJj...


In [7]:
visual_df = df[["movie_id", "title", "poster_path"]].dropna(subset=["poster_path"])
visual_df = visual_df[visual_df["poster_path"].str.startswith("http")].reset_index(drop=True)

visual_df.to_csv(f"{PROCESSED_DIR}/visual_posters.csv", index=False)
print(f"visual_posters.csv saved — {len(visual_df)} movies with posters")
visual_df.head(3)

visual_posters.csv saved — 9988 movies with posters


,movie_id,title,poster_path
0,983044,The Arctic Convoy,https://image.tmdb.org/t/p/w500/7qtd6xschdz7GB...
1,5,Four Rooms,https://image.tmdb.org/t/p/w500/75aHn1NOYXh4M7...
2,11,Star Wars,https://image.tmdb.org/t/p/w500/6FfCtAuVAW8XJj...


In [8]:
C = df["vote_average"].mean()
m = df["vote_count"].quantile(0.70)

print(f"C (mean rating)  : {C:.3f}")
print(f"m (min votes 70%): {m:.0f}")

def weighted_rating(row, m=m, C=C):
    v = row["vote_count"]
    R = row["vote_average"]
    return (v / (v + m) * R) + (m / (v + m) * C)

df["weighted_score"] = df.apply(weighted_rating, axis=1)

popularity_df = df[[
    "movie_id", "title", "vote_average", "vote_count",
    "popularity", "weighted_score", "genres", "release_year",
    "poster_path", "overview"
]].sort_values("weighted_score", ascending=False).reset_index(drop=True)

popularity_df.to_csv(f"{PROCESSED_DIR}/popularity.csv", index=False)
print(f"\npopularity.csv saved")
print(f"Top 5 movies by weighted score:")
popularity_df[["title", "vote_average", "weighted_score"]].head()

C (mean rating)  : 6.655
m (min votes 70%): 1723

popularity.csv saved
Top 5 movies by weighted score:


,title,vote_average,weighted_score
0,The Shawshank Redemption,8.719,8.607825
1,The Godfather,8.700,8.556630
2,The Dark Knight,8.500,8.414918
3,Schindler's List,8.568,8.395597
4,Interstellar,8.471,8.395250


In [9]:
links_path   = f"{MOVIELENS_DIR}/links.csv"
ratings_path = f"{MOVIELENS_DIR}/ratings.csv"

if not os.path.exists(links_path) or not os.path.exists(ratings_path):
    print("MovieLens files not found yet — skip for now")
    print(f"Download from: https://grouplens.org/datasets/movielens/25m/")
    print(f"Put in: {MOVIELENS_DIR}/")
else:
    links   = pd.read_csv(links_path)[["movieId", "tmdbId"]].dropna()
    links["tmdbId"] = links["tmdbId"].astype(int)

    print("Loading ratings...")
    ratings = pd.read_csv(ratings_path,
                          usecols=["userId", "movieId", "rating"],
                          dtype={"userId": int, "movieId": int, "rating": float})

    ratings  = ratings.merge(links, on="movieId")
    tmdb_ids = set(df["movie_id"].astype(int))
    ratings  = ratings[ratings["tmdbId"].isin(tmdb_ids)]

    user_counts  = ratings["userId"].value_counts()
    active_users = user_counts[user_counts >= 20].index
    ratings      = ratings[ratings["userId"].isin(active_users)]

    collab_df = ratings[["userId", "tmdbId", "rating"]].rename(
        columns={"tmdbId": "movie_id"}
    ).reset_index(drop=True)

    collab_df.to_csv(f"{PROCESSED_DIR}/collaborative_ratings.csv", index=False)
    print(f"collaborative_ratings.csv saved")
    print(f"Ratings: {len(collab_df)} | Users: {collab_df['userId'].nunique()}")


Loading ratings...
collaborative_ratings.csv saved
Ratings: 21451796 | Users: 156066


In [10]:
df.to_csv(f"{PROCESSED_DIR}/movies_master.csv", index=False)

print("=" * 45)
print("  Preprocessing Complete!")
print("=" * 45)
for f in sorted(os.listdir(PROCESSED_DIR)):
    size = os.path.getsize(f"{PROCESSED_DIR}/{f}") / 1024
    print(f"  {f:40s} {size:8.1f} KB")

  Preprocessing Complete!
  collaborative_ratings.csv                340221.2 KB
  content_based.csv                          7529.0 KB
  metadata.csv                               4883.3 KB
  movies_master.csv                         16552.2 KB
  popularity.csv                             4110.6 KB
  visual_posters.csv                          861.0 KB
